In [1]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
# Load data

%load_ext autoreload
%autoreload 2

import pandas as pd
from churnguard.data import PROJECT_ROOT
from churnguard.features import (
    NUMERIC_FEATURES, build_preprocessor, engineer_features, split_X_y,
)

train = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "test.parquet")

X_train, y_train = split_X_y(train)
X_test, y_test = split_X_y(test)
print(X_train.shape, X_test.shape)

(5634, 19) (1409, 19)


In [3]:
# Do the engineered features carry signal?

eng = engineer_features(train)

display(eng.groupby("num_addon_services")["churned"].agg(["mean", "count"]).round(3))
display(eng.groupby("is_auto_pay")["churned"].agg(["mean", "count"]).round(3))
display(
    eng.groupby(pd.qcut(eng["charge_diff"], 5, duplicates="drop"), observed=True)["churned"]
       .agg(["mean", "count"]).round(3)
)
print("Remaining NaN in total_charges:", eng["total_charges"].isna().sum())

,mean,count
num_addon_services,,
0,0.209,1757
1,0.458,766
2,0.362,832
3,0.278,885
4,0.230,695
5,0.125,464
6,0.047,235


,mean,count
is_auto_pay,,
0,0.350,3177
1,0.155,2457


,mean,count
charge_diff,,
"(-18.901, -1.531]",0.308,1127
"(-1.531, -0.3]",0.165,1127
"(-0.3, 0.276]",0.369,1126
"(0.276, 1.528]",0.178,1127
"(1.528, 19.125]",0.306,1127


Remaining NaN in total_charges: 0


In [4]:
# Fit on train, transform both

pre = build_preprocessor()

Xt_train = pre.fit_transform(X_train)  # LEARN from train
Xt_test = pre.transform(X_test)        # only APPLY to test

print(Xt_train.shape, Xt_test.shape)
print("NaNs:", Xt_train.isna().sum().sum(), Xt_test.isna().sum().sum())
print(Xt_train.columns.tolist())
Xt_train.head()

(5634, 37) (1409, 37)
NaNs: 0 0
['tenure', 'monthly_charges', 'total_charges', 'num_addon_services', 'charge_diff', 'senior_citizen', 'is_auto_pay', 'contract', 'partner_No', 'partner_Yes', 'dependents_No', 'dependents_Yes', 'phone_service_No', 'phone_service_Yes', 'multiple_lines_No', 'multiple_lines_Yes', 'internet_service_DSL', 'internet_service_Fiber optic', 'internet_service_No', 'online_security_No', 'online_security_Yes', 'online_backup_No', 'online_backup_Yes', 'device_protection_No', 'device_protection_Yes', 'tech_support_No', 'tech_support_Yes', 'streaming_tv_No', 'streaming_tv_Yes', 'streaming_movies_No', 'streaming_movies_Yes', 'paperless_billing_No', 'paperless_billing_Yes', 'payment_method_Bank transfer (automatic)', 'payment_method_Credit card (automatic)', 'payment_method_Electronic check', 'payment_method_Mailed check']


,tenure,monthly_charges,total_charges,num_addon_services,charge_diff,senior_citizen,is_auto_pay,contract,partner_No,partner_Yes,...,streaming_tv_No,streaming_tv_Yes,streaming_movies_No,streaming_movies_Yes,paperless_billing_No,paperless_billing_Yes,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
0,0.102371,-0.521976,-0.262257,0.507935,0.226917,0,0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-0.711743,0.337478,-0.503635,-0.570530,-0.639564,0,0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,-0.793155,-0.809013,-0.749883,0.507935,-1.867855,0,0,2.0,0.0,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-0.263980,0.284384,-0.172722,1.047168,0.081601,0,1,2.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,-1.281624,-0.676279,-0.989374,-1.109762,0.003149,0,0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [5]:
# What "fit on train only" looks like

pd.DataFrame({
    "train_mean": Xt_train[NUMERIC_FEATURES].mean(),
    "test_mean": Xt_test[NUMERIC_FEATURES].mean(),
    "train_std": Xt_train[NUMERIC_FEATURES].std(),
}).round(3)

,train_mean,test_mean,train_std
tenure,-0.0,-0.023,1.0
monthly_charges,-0.0,-0.028,1.0
total_charges,0.0,-0.043,1.0
num_addon_services,0.0,-0.054,1.0
charge_diff,-0.0,0.013,1.0


In [6]:
# Robustness to unseen values

weird = X_test.iloc[[0]].copy()
weird["payment_method"] = "Crypto wallet"  # never seen in training
weird["contract"] = "Three year"           # never seen in training

out = pre.transform(weird)
out[["contract"] + [c for c in out.columns if c.startswith("payment_method")]]

,contract,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
0,-1.0,0.0,0.0,0.0,0.0
